In [103]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np
import pandas as pd


In [104]:
# Load the trained model,scaler, and one-hot encoder
model = load_model('model.h5')
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# load encoder and scaler
with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)

with open('one_hot_encoder_geo.pkl', 'rb') as f:
    one_hot_encoder_geo = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [105]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000 
}
print(label_encoder_gender.classes_)
print(one_hot_encoder_geo.categories_)

['Female' 'Male']
[array(['France', 'Germany', 'Spain'], dtype=object)]


In [106]:
# One-hot encode 'Geography'
geo_encoded = one_hot_encoder_geo.transform(np.array([input_data['Geography']]).reshape(-1, 1))
geo_encoded_df = pd.DataFrame(geo_encoded, columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))

print(geo_encoded_df)

   Geography_France  Geography_Germany  Geography_Spain
0               1.0                0.0              0.0


c:\dev\Python\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [107]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [108]:
# Encode categorical variables
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [109]:
# Concatenate the one-hot encoded columns with the original DataFrame
input_df = pd.concat([input_df.drop('Geography',axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [110]:
# Scaling the input data
input_scaled = scaler.transform(input_df, copy=False)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [111]:
# Predict Churn
predictions = model.predict(input_scaled)
predictions 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step


array([[0.0499193]], dtype=float32)

In [112]:
prediction_proba = predictions[0][0]
prediction_proba

0.049919304

In [113]:
if prediction_proba > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is unlikely to churn.")

The customer is unlikely to churn.


Bu kod, bir müşterinin verilerini alarak churn (müşteri kaybı) tahmini yapar. İşte özet:

Model ve Dönüştürücülerin Yüklenmesi:

Eğitim sırasında kaydedilen model (model.h5), etiket dönüştürücüler (label_encoder_gender.pkl, one_hot_encoder_geo.pkl) ve ölçekleyici (scaler.pkl) yüklenir.
Giriş Verisinin Hazırlanması:

Örnek müşteri verisi (input_data) tanımlanır.
Geography sütunu one-hot encoding ile dönüştürülür.
Gender sütunu label encoding ile sayısal bir değere dönüştürülür.
Dönüştürülmüş sütunlar orijinal veriyle birleştirilir.
Özelliklerin Ölçeklenmesi:

Giriş verisi, eğitim sırasında kullanılan ölçekleyici (scaler) ile standartlaştırılır.
Tahmin Yapılması:

Model, ölçeklenmiş veriyi kullanarak churn olasılığını tahmin eder.
Tahmin edilen olasılık (prediction_proba) 0.5'ten büyükse müşteri kaybı olasıdır, aksi takdirde değildir.
Sonuç:

Tahmin sonucu ekrana yazdırılır: "The customer is likely to churn." veya "The customer is unlikely to churn."